In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

flights = pd.read_csv("../data/cleaned/cleaned_flights.csv")

In [ ]:
import statsmodels.formula.api as smf

logit_model = smf.logit("delayed_15 ~ departure_hour + C(carrier_code) + C(day_of_week) + C(month) + C(origin_airport)", data=flights).fit()

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "Linear Model 1",
        "Linear Model 2",
        "Logistic Model"
    ],
    "Response Variable": [
        "Departure Delay",
        "Departure Delay",
        "Delayed 15+"
    ],
    "Predictors": [
        "Departure Hour",
        "Hour + Airline + Day + Month + Airport",
        "Hour + Airline + Day + Month + Airport"
    ],
    "Metric": [
        "R^2",
        "R^2",
        "Pseudo R^2"
    ],
    "Fit Statistic": [
        0.001,
        0.008,
        0.019
    ]
})

comparison

The first model shows that departure hour alone does not explain much of the variation in departure delays ($R^2 = 0.001$). Adding airline, day of week, month, and airport improves the model ($R^2 = 0.008$), showing that these factors help explain delays. The logistic regression model had a pseudo $R^2$ of 0.019, suggesting that these variables are somewhat better at predicting whether a flight will be delayed by at least 15 minutes than predicting the exact length of the delay.

In [ ]:
prob_delay = (flights.groupby("carrier_code")["delayed_15"].mean().sort_values())

prob_delay.plot(kind="bar")

plt.ylabel("Probability of 15+ Minute Delay")
plt.title("Probability of Delay by Airline")

plt.show()

United Airlines exhibited the lowest probability of a 15 minute departure delay, while American Airlines exhibited the highest probability among the airlines included in this study.

In [ ]:
odds_df = pd.DataFrame({
    "Coefficient": logit_model.params,
    "Odds Ratio": np.exp(logit_model.params)
})

odds_df

Odds ratios were calculated to make the logistic regression results easier to interpret. An odds ratio greater than 1 indicates increased delay risk, while an odds ratio less than 1 indicates decreased delay risk relative to the reference.

The odds ratio for departure hour was 1.042, indicating that each additional hour later in the day increases the odds of a flight being delayed by at least 15 minutes by about 4.2%. Relative to American Airlines, United Airlines had substantially lower odds of experiencing a delay (0.67). Sunday flights had around 11% higher odds of delay than Friday flights, while Tuesday flights had about 26% lower odds. July and December have the highest delay risk, with odds ratios of 1.51 and 1.55, respectively, compared to January. Airport effects were small, with SFO showing around 5% higher odds of delay than LAX and SAN showing 2.5% lower odds of delay than LAX.